# Top-k routing and token dispatch

This notebook builds a trainable sigmoid router and runs sparse token dispatch through the expert bank.

In [ ]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(17)

In [ ]:
from src.model import DeepSeekConfig, TopKRouter

config = DeepSeekConfig()
router = TopKRouter(config)
x = torch.randn(2, 4, config.dim)
indices, weights = router(x)

print("Indices:", indices.shape)
print("Weights:", weights.shape)
print("Token 0 IDs:", indices[0, 0].tolist())
print("Token 0 weights:", weights[0, 0].tolist())
print("Token 0 weight sum:", weights[0, 0].sum().item())

In [ ]:
from src.model import DeepSeekMoE

moe = DeepSeekMoE(config)
output = moe(x)
print("Input:", x.shape)
print("Output:", output.shape)
print("MoE parameters:", sum(p.numel() for p in moe.parameters()))
assert output.shape == x.shape

In [ ]:
flat_x = x.reshape(-1, config.dim)
indices, weights = moe.router(flat_x)
counts = torch.bincount(indices.reshape(-1), minlength=config.n_routed_experts)
print("Assignments:", int(counts.sum()))
for expert_id, count in enumerate(counts.tolist()):
    print(f"Expert {expert_id}: {count}")

In [ ]:
x_with_grad = torch.randn(2, 4, config.dim, requires_grad=True)
moe_with_grad = DeepSeekMoE(config)
moe_with_grad(x_with_grad).square().mean().backward()
print("Router gradient:", moe_with_grad.router.gate.weight.grad.norm().item())